# IKEA Fire Detection — Sensor Fusion Model Training

Trains a small CNN that takes a 32×32 grayscale image + 1 temperature value
and classifies the scene as `fire` or `safe_environment`.

Dataset collected manually with Arduino Nano 33 BLE Sense + OV7675 camera + DHT20.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Path to your zipped dataset on Google Drive
# Upload dataset.zip to your Drive's root, or change this path
DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'

# Unzip into Colab's local /content folder for fast access
!rm -rf /content/dataset
!unzip -q "$DATASET_ZIP" -d /content/
!ls /content/dataset

In [ ]:
!pip install edgeimpulse --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, Flatten, Concatenate,
                                     Conv2D, MaxPooling2D, Dropout, Lambda)
import tensorflow as tf
import edgeimpulse as ei

# ==== Configuration ====
BASE_DIR     = '/content/dataset'
CSV_PATH     = os.path.join(BASE_DIR, 'temperatures.csv')

IMG_WIDTH    = 32
IMG_HEIGHT   = 32
IMG_CHANNELS = 1   # grayscale

CLASSES      = ['safe_environment', 'fire']

# Edge Impulse API key — get from your project Dashboard → Keys
ei.API_KEY = "ei_PASTE_YOUR_KEY_HERE"

print('Config loaded.')
print(f'Image size: {IMG_WIDTH}x{IMG_HEIGHT}x{IMG_CHANNELS}')
print(f'Classes: {CLASSES}')

In [ ]:
data = pd.read_csv(CSV_PATH)
print(f'Total rows in CSV: {len(data)}')
print(f'\nSamples per class:')
print(data['class'].value_counts())
print(f'\nFirst few rows:')
data.head()

In [ ]:
def load_and_preprocess_image(filepath):
    """Load PNG → grayscale → resize to 32x32 → normalize 0..1"""
    if IMG_CHANNELS == 1:
        img = Image.open(filepath).convert('L')  # grayscale
    else:
        img = Image.open(filepath).convert('RGB')
    img = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    if IMG_CHANNELS == 1:
        arr = arr.reshape(IMG_HEIGHT, IMG_WIDTH, 1)
    return arr

images = []
valid_rows = []

for idx, row in data.iterrows():
    filepath = os.path.join(BASE_DIR, row['class'], row['filename'])
    if not os.path.exists(filepath):
        print(f'  ! missing: {filepath}')
        continue
    images.append(load_and_preprocess_image(filepath))
    valid_rows.append(idx)

images = np.array(images)
data = data.loc[valid_rows].reset_index(drop=True)

print(f'\nLoaded {len(images)} images, shape: {images.shape}')
print(f'Pixel range: {images.min():.3f} to {images.max():.3f}')
print(f'CSV rows now: {len(data)}')

assert len(images) == len(data), 'Image count must match CSV row count!'

In [ ]:
# Encode 'safe_environment'/'fire' as 0/1
label_encoder = LabelEncoder()
data['label_encoded'] = label_encoder.fit_transform(data['class'])
num_classes = len(label_encoder.classes_)

print(f'Class mapping:')
for i, c in enumerate(label_encoder.classes_):
    print(f'  {i} → {c}')

# Get temperatures and labels
temperatures = data['temperature'].values.reshape(-1, 1).astype(np.float32)
labels = to_categorical(data['label_encoded'].values, num_classes=num_classes)

# Flatten images and concatenate temperature as last feature
flattened_images = images.reshape(images.shape[0], -1)
combined_inputs = np.concatenate([flattened_images, temperatures], axis=1).astype(np.float32)

print(f'\nFlattened images shape: {flattened_images.shape}')
print(f'Temperatures shape: {temperatures.shape}')
print(f'Combined inputs shape: {combined_inputs.shape}  ← this is what the model takes')
print(f'Labels shape: {labels.shape}')

print(f'\nTemperature stats per class:')
for c in label_encoder.classes_:
    mask = data['class'] == c
    temps = data.loc[mask, 'temperature']
    print(f'  {c}: mean={temps.mean():.1f}°C, range={temps.min():.1f}-{temps.max():.1f}°C')

In [ ]:
def build_sensor_fusion_model():
    """
    Takes one flat input vector of length (W*H*C + 1).
    Splits it inside the model into an image branch (CNN) and
    a temperature branch (Dense), then merges them.
    """
    total_features = IMG_WIDTH * IMG_HEIGHT * IMG_CHANNELS + 1
    input_layer = Input(shape=(total_features,))

    # Slice into image features and temperature
    image_part = Lambda(lambda x: x[:, :-1])(input_layer)
    temp_part  = Lambda(lambda x: x[:, -1:])(input_layer)

    # Reshape image back to (32, 32, 1)
    image_part = Lambda(
        lambda x: tf.reshape(x, [-1, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
    )(image_part)

    # ==== Image branch (CNN) ====
    x = Conv2D(32, (3, 3), activation='relu')(image_part)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)

    # ==== Temperature branch ====
    y = Dense(32, activation='relu')(temp_part)

    # ==== Fusion ====
    combined = Concatenate()([x, y])
    z = Dense(128, activation='relu')(combined)
    z = Dropout(0.5)(z)
    z = Dense(num_classes, activation='softmax')(z)

    model = Model(inputs=input_layer, outputs=z)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_sensor_fusion_model()
model.summary()

In [ ]:
# 80/20 split for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    combined_inputs, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

# Train!
EPOCHS = 50
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy * 100:.2f}%')

# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def classify_sample(image_path, temperature):
    """Run inference on one image+temperature pair."""
    img = load_and_preprocess_image(image_path).flatten()
    temp = np.array([temperature], dtype=np.float32)
    combined = np.concatenate([img, temp]).reshape(1, -1)

    prediction = model.predict(combined, verbose=0)[0]
    predicted_idx = np.argmax(prediction)
    predicted_label = label_encoder.inverse_transform([predicted_idx])[0]

    print(f'Image: {image_path}')
    print(f'Temperature: {temperature}°C')
    print(f'Confidence scores:')
    for i, score in enumerate(prediction):
        label = label_encoder.inverse_transform([i])[0]
        print(f'  {label}: {score:.4f}')
    print(f'→ Predicted: {predicted_label} ({prediction[predicted_idx]*100:.1f}% confidence)\n')
    return predicted_label, prediction

# Test on a known SAFE sample
safe_files = os.listdir(os.path.join(BASE_DIR, 'safe_environment'))
test_safe = os.path.join(BASE_DIR, 'safe_environment', safe_files[0])
classify_sample(test_safe, temperature=23.0)

# Test on a known FIRE sample
fire_files = os.listdir(os.path.join(BASE_DIR, 'fire'))
test_fire = os.path.join(BASE_DIR, 'fire', fire_files[0])
classify_sample(test_fire, temperature=32.0)

# Optional: test that temperature actually matters
# Pass a FIRE image with SAFE temperature — model should ideally still predict fire
# (because the image is the strongest signal) but with lower confidence
print('=== Sensor fusion test: fire image + low temp ===')
classify_sample(test_fire, temperature=22.0)

In [ ]:
# Check that the model will fit in the Nano's RAM/Flash
try:
    profile = ei.model.profile(model=model, device='arduino-nano-33-ble')
    print(profile.summary())
except Exception as e:
    print(f'Could not profile: {e}')

In [ ]:
# Choose your deployment target
deploy_target = 'arduino'

# Build and download the Arduino library
output = ei.model.deploy(
    model=model,
    model_output_type=ei.model.output_type.Classification(labels=list(label_encoder.classes_)),
    deploy_target=deploy_target,
    engine='tflite-eon'  # EON Compiler saves ~18% RAM
)

# Save the deployed library to /content
output_zip_path = '/content/ei-model-arduino.zip'
with open(output_zip_path, 'wb') as f:
    f.write(output)

print(f'\n✓ Deployed library saved to: {output_zip_path}')
print('Now download it using the next cell.')

In [ ]:
from google.colab import files
files.download('/content/ei-model-arduino.zip')

In [ ]:
import shutil
shutil.copy('/content/ei-model-arduino.zip', '/content/drive/MyDrive/ei-model-arduino.zip')
print('Backup saved to Drive.')

# Also save the trained Keras model in case you want to retrain later
model.save('/content/drive/MyDrive/fire_detection_model.keras')
print('Keras model saved.')